In [ ]:
from __future__ import annotations

import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp

# 3.1. One-Dimensional State Space Form

In [ ]:
m = 1.5  # kg (effective mass)
b = 15.0  # N·s/m (damping)
k = 200.0  # N/m (stiffness)

In [ ]:
# TODO. Derive the state space matrices
A = np.zeros((2, 2))  # placeholder for A matrix
B = np.zeros((2, 1))  # placeholder for B matrix
C = np.zeros((1, 2))  # placeholder for C matrix

print("A:\n", A)
print("B:\n", B)
print("C:\n", C)

In [ ]:
D = np.array([[0]])  # no direct feedthrough

# TODO. Set the time domain and initial conditions
# Then use signal.lsim to compute the response of the system to a step input

In [ ]:
# TODO. Plot the results.
# Recommended: Create two subplots one for position and one for velocity

In [ ]:
# Parameter sets
bs = [5.0, 15.0, 40.0]

# Simulate the system for each parameter set and store the results
for b in bs:
    # Update the B matrix for the current damping coefficient
    # TODO. Update the B matrix
    
    # Simulate the system response to a step input
    # TODO. Simulate the system response using signal.lsim or solve_ivp
    pass


# TODO. Plot the results

# 3.2. Two-Dimensional State Space Form

## 3.2.1. Derivation and Simulation

In [ ]:
import numpy as np

M_2d = np.array([[1.5, 0.3], [0.3, 2.0]])
B_2d = np.array([[15.0, 3.0], [3.0, 20.0]])
K_2d = np.array([[200.0, 50.0], [50.0, 350.0]])

In [ ]:
# TODO. Derive the state space matrices for the 2D system.
# Hint: The state vector should include both positions and velocities of the two masses.

A = np.block(
    [
        [np.zeros((2, 2)), np.zeros((2, 2))],
        [np.zeros((2, 2)), np.zeros((2, 2))],
    ]
)  # placeholder for A matrix
B = np.block([[np.zeros((2, 2))], [np.zeros((2, 2))]])  # placeholder for B matrix
C = np.block([[np.zeros((2, 2)), np.zeros((2, 2))]])  # placeholder for C matrix

print("A:\n", A)
print("B:\n", B)
print("C:\n", C)

In [ ]:
# TODO. Set the time domain and initial conditions
# Then use signal.lsim to compute the response of the system to a step input

In [ ]:
# TODO. Plot the results.
# - Hand trajectory in the xy-plane
# - Position components vs time
# - Speed profile |\dot{p}(t)| vs time


## 3.2.2. Smooth Reaching with a Minimum-Jerk Profile

In [ ]:
def min_jerk_profile(t: np.ndarray, T: float) -> np.ndarray:
    """Minimum-jerk interpolation profile s(t) in [0, 1].

    s(t) = 10(t/T)^3 - 15(t/T)^4 + 6(t/T)^5   for 0 <= t <= T
    s(t) = 1                                      for t > T

    Args:
        t: Time array.
        T: Reach duration.

    Returns:
        s: Profile values, same shape as t.
    """
    tau = np.clip(t / T, 0.0, 1.0)
    return 10 * tau**3 - 15 * tau**4 + 6 * tau**5


def simulate_smooth_reaching(
    M: np.ndarray,
    B: np.ndarray,
    K: np.ndarray,
    p_start: np.ndarray,
    p_target: np.ndarray,
    T_reach: float = 0.5,
    T_total: float = 1.0,
    dt: float = 1 / 60,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Simulate a reaching movement with minimum-jerk equilibrium shift.

    The equilibrium moves as: p_eq(t) = p_start + s(t) * (p_target - p_start)
    where s(t) is the minimum-jerk profile.

    Args:
        M, B, K:    Impedance matrices (n, n).
        p_start:    Start position (n,).
        p_target:   Target position (n,).
        T_reach:    Duration of the equilibrium shift.
        T_total:    Total simulation time.
        dt:         Output time step.

    Returns:
        t:   Time array, shape (T_steps,).
        pos: Position array, shape (T_steps, n).
        vel: Velocity array, shape (T_steps, n).
    """
    M, B, K = np.atleast_2d(M), np.atleast_2d(B), np.atleast_2d(K)
    p_start, p_target = np.asarray(p_start, float), np.asarray(p_target, float)
    n = M.shape[0]
    Mi = np.linalg.inv(M)
    direction = p_target - p_start

    def dynamics(t_val, state):
        pos = state[:n]
        vel = state[n:]
        s = float(min_jerk_profile(np.array([t_val]), T_reach)[0])

        # TODO. Edit the following line to compute equilibrium position p_eq
        p_eq = np.zeros_like(pos)
        delta_p = pos - p_eq

        # TODO. Replace the following line with the correct acceleration computation
        acc = np.zeros_like(vel)
        return np.concatenate([vel, acc])

    x0 = np.concatenate([p_start, np.zeros(n)])
    t_eval = np.arange(0, T_total + dt / 2, dt)
    sol = solve_ivp(dynamics, [0, T_total], x0, t_eval=t_eval, max_step=dt)

    t = sol.t
    pos = sol.y[:n].T
    vel = sol.y[n:].T
    return t, pos, vel

In [ ]:
# TODO. Simulate a reaching movement and plot the results.
p_start = np.array([0.30, 0.20])
p_target = np.array([0.45, 0.35])

t_mj, pos_mj, vel_mj = simulate_smooth_reaching(
    M_2d, B_2d, K_2d, p_start, p_target, T_reach=0.5, T_total=1.0
)

In [ ]:
# TODO. Plot:
# - Hand trajectory in the xy-plane
# - Position components vs time
# - Speed profile



# 3.3. Three-Dimensional State Space Form

In [ ]:
import numpy as np

M_3d = np.array([[1.5, 0.3, 0.0], [0.3, 2.0, 0.0], [0.0, 0.0, 1.0]])
B_3d = np.array([[15.0, 4.0, 0.0], [4.0, 25.0, 0.0], [0.0, 0.0, 10.0]])
K_3d = np.array([[200.0, 60.0, 0.0], [60.0, 400.0, 0.0], [0.0, 0.0, 100.0]])

In [ ]:
# TODO. Simulate a reaching movement and plot the results.

In [ ]:
# TODO. Plot the results.
# - Hand trajectory in the xy-plane
# - Position components vs time
# - Speed profile |\dot{p}(t)| vs time
